# Chapter 9 — Multimodal Large Language Models
## Solutions Notebook

**Book:** Hands-On Large Language Models — Jay Alammar & Maarten Grootendorst (O'Reilly)  
**Chapter:** 9 — Multimodal Large Language Models  
**Companion notes:** `notes/ch09-multimodal-llms.md`

---

> This is your personal solutions file. Write your solutions here.  
> The template file in `template/` stays untouched.

In [ ]:
# Optional: uncomment to install dependencies in Colab
# !pip install transformers sentence-transformers open_clip_torch torch torchvision pillow requests matplotlib

---
## Part 0 — Theory Warm-Up

---
### T1 — Image Patch Tokenization from Scratch

In [ ]:
import numpy as np

def image_to_patches(image, patch_size):
    """
    Split a 2D greyscale image into non-overlapping square patches and flatten each.

    Args:
        image:      numpy array of shape (H, W)
        patch_size: integer side length of each square patch

    Returns:
        patches: numpy array of shape (num_patches, patch_size * patch_size)
    """
    H, W = image.shape
    print(f"Image shape: {image.shape}")

    # Number of patches along each dimension
    n_h = H // patch_size
    n_w = W // patch_size
    print(f"Patch grid: {n_h} × {n_w} = {n_h * n_w} patches")
    print(f"Each patch flattened: {patch_size * patch_size} values")

    # Reshape into patch grid: (n_h, patch_size, n_w, patch_size)
    # then move axes to (n_h, n_w, patch_size, patch_size)
    # then flatten patches and merge n_h*n_w into one dimension
    patches = (
        image
        .reshape(n_h, patch_size, n_w, patch_size)
        .transpose(0, 2, 1, 3)       # → (n_h, n_w, patch_size, patch_size)
        .reshape(n_h * n_w, patch_size * patch_size)
    )

    print(f"Output shape: {patches.shape}")
    return patches


# Test with a 28×28 random image and patch_size=4
np.random.seed(42)
fake_image = np.random.randint(0, 256, size=(28, 28), dtype=np.uint8)

patches = image_to_patches(fake_image, patch_size=4)

assert patches.shape == (49, 16), f"Expected (49, 16), got {patches.shape}"
print("T1 passed — patches shape:", patches.shape)

---
### T2 — Contrastive Similarity Matrix from Scratch

In [ ]:
import numpy as np

def cosine_similarity_matrix(image_embs, text_embs):
    """
    Compute an N×N cosine similarity matrix between image and text embeddings.

    Args:
        image_embs: numpy array of shape (N, D)
        text_embs:  numpy array of shape (N, D)

    Returns:
        sim_matrix: numpy array of shape (N, N) where [i, j] = sim(text_i, image_j)
    """
    # L2-normalise: each row becomes a unit vector
    # After normalisation, dot product = cosine similarity
    img_norms = np.linalg.norm(image_embs, axis=-1, keepdims=True)
    txt_norms = np.linalg.norm(text_embs, axis=-1, keepdims=True)

    image_embs_norm = image_embs / img_norms   # shape: (N, D)
    text_embs_norm  = text_embs  / txt_norms   # shape: (N, D)

    # sim[i, j] = dot(text_i, image_j) = cosine similarity
    # text_embs_norm @ image_embs_norm.T gives (N, N)
    sim_matrix = text_embs_norm @ image_embs_norm.T

    return sim_matrix


# Test with 3 paired embeddings (from notes section 4e dry-run)
image_embs = np.array([
    [0.90,  0.44],   # puppy image
    [0.10,  0.99],   # cat image
    [0.71, -0.71],   # car image
], dtype=np.float32)

text_embs = np.array([
    [0.85,  0.53],   # puppy caption
    [0.05,  1.00],   # cat caption
    [0.71, -0.70],   # car caption
], dtype=np.float32)

sim = cosine_similarity_matrix(image_embs, text_embs)

print("Similarity matrix (rows=text, cols=image):")
labels = ["puppy", "cat", "car"]
print(f"{'':12}" + "".join(f"{l:>10}" for l in labels))
for i, row_label in enumerate(labels):
    print(f"{row_label:12}" + "".join(f"{sim[i,j]:>10.3f}" for j in range(3)))

# Verify: diagonal should be max in each row
for i in range(3):
    assert sim[i, i] == sim[i].max(), f"Row {i}: diagonal is not max!"

print("T2 passed — diagonal is highest in every row")

---
## Part 1 — Vision Transformer Concepts

---
### 1.1 — Load an Image and Split It into Patches

In [ ]:
from PIL import Image
from urllib.request import urlopen
import numpy as np
import matplotlib.pyplot as plt

IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg"

# Load the image and convert to 28×28 greyscale
raw_image = Image.open(urlopen(IMG_URL)).convert("L")   # L = greyscale
image_28 = raw_image.resize((28, 28))
image_array = np.array(image_28)
print(f"Original size: {raw_image.size}")
print(f"After resize:  {image_array.shape}")

# Split into 4×4 grid of 7×7 patches → 16 patches
patches = image_to_patches(image_array, patch_size=7)
print(f"Patches shape: {patches.shape}")

# Visualise patches in a 4×4 grid
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for idx, ax in enumerate(axes.flat):
    # Reshape each flattened patch back to (7, 7) for display
    patch_img = patches[idx].reshape(7, 7)
    ax.imshow(patch_img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"P{idx+1}", fontsize=8)
    ax.axis("off")

plt.suptitle("16 patches from the 28×28 image (each 7×7 pixels)", fontsize=10)
plt.tight_layout()
plt.show()

---
### 1.2 — Linear Projection of Patch Embeddings

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# Convert patches to a float tensor — shape (16, 49)
patches_tensor = torch.from_numpy(patches).float()
print(f"patches_tensor shape: {patches_tensor.shape}")
# → (16, 49): 16 patches, each with 7×7=49 raw pixel values

# Define the linear projection: maps each 49-dim patch to an 8-dim embedding
# This is the W_p matrix from the notes (section 2c)
projection = nn.Linear(in_features=49, out_features=8)

# Apply projection
patch_embeddings = projection(patches_tensor)
print(f"After projection:     {patch_embeddings.shape}")
# → (16, 8): 16 patch embeddings, each 8-dimensional

# Prepend a [CLASS] token (zero-initialised at the start of training)
# In real ViT this is a learnable parameter, not hardcoded zeros
cls_token = torch.zeros(1, 8)
sequence = torch.cat([cls_token, patch_embeddings], dim=0)
print(f"After CLS prepend:    {sequence.shape}")
# → (17, 8): 17 tokens (1 CLS + 16 patches), each 8-dimensional

print()
print("Interpretation:")
print("  17 tokens = 1 [CLASS] token + 16 patch tokens")
print("  8 = d_model (the embedding dimension the Transformer will process)")
print("  This sequence is passed to the Transformer encoder unchanged.")

---
## Part 2 — OpenCLIP

---
### 2.1 — Load the CLIP Model

In [ ]:
from transformers import CLIPTokenizerFast, CLIPProcessor, CLIPModel

MODEL_ID = "openai/clip-vit-base-patch32"

# Tokenizer converts text to token IDs (identical to BERT-style tokenization)
clip_tokenizer = CLIPTokenizerFast.from_pretrained(MODEL_ID)

# Processor handles image preprocessing: resize to 224×224, normalise pixel values
clip_processor = CLIPProcessor.from_pretrained(MODEL_ID)

# Main model: contains both the ViT image encoder and the text Transformer encoder
clip_model = CLIPModel.from_pretrained(MODEL_ID)
clip_model.eval()   # inference mode — disable dropout

print("Tokenizer type:", type(clip_tokenizer).__name__)
print("Processor type:", type(clip_processor).__name__)
print("Model type:    ", type(clip_model).__name__)

---
### 2.2 — Generate Text Embeddings

In [ ]:
import torch

captions = [
    "A puppy playing in the snow",
    "A pixelated image of a cute cat",
    "A supercar on the road with sunset in background",
]

# Tokenize all 3 captions in one batch call
# padding=True ensures all sequences are padded to the same length
text_inputs = clip_tokenizer(captions, padding=True, return_tensors="pt")
print("Input IDs shape:", text_inputs["input_ids"].shape)
# → (3, max_caption_length) — one row per caption

# Extract text embeddings — CLIP's text encoder output (pooled)
with torch.no_grad():
    text_embs_clip = clip_model.get_text_features(**text_inputs)

print("Raw text embeddings shape:", text_embs_clip.shape)
# → torch.Size([3, 512]) — 3 captions, each 512-dimensional

# L2-normalise so dot product = cosine similarity
text_embs_clip = text_embs_clip / text_embs_clip.norm(dim=-1, keepdim=True)
print("Normalised — all norms should be 1.0:")
print(text_embs_clip.norm(dim=-1))   # → tensor([1., 1., 1.])

---
### 2.3 — Generate Image Embeddings

In [ ]:
from PIL import Image
from urllib.request import urlopen
import torch

image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/2009_Audi_TT_--_10-30-2011.jpg/320px-2009_Audi_TT_--_10-30-2011.jpg",
]
image_labels = ["puppy", "cat", "car"]

# Load all 3 images as PIL Image objects
images = [
    Image.open(urlopen(url)).convert("RGB")
    for url in image_urls
]
print(f"Loaded {len(images)} images")
for label, img in zip(image_labels, images):
    print(f"  {label}: {img.size}")

# Preprocess all 3 images in one batch
# The processor resizes each to 224×224 and normalises pixel values
image_inputs = clip_processor(text=None, images=images, return_tensors="pt")
print("pixel_values shape:", image_inputs["pixel_values"].shape)
# → torch.Size([3, 3, 224, 224]) — 3 images, RGB, 224×224

# Extract image embeddings using CLIP's ViT image encoder
with torch.no_grad():
    image_embs_clip = clip_model.get_image_features(**image_inputs)

print("Raw image embeddings shape:", image_embs_clip.shape)
# → torch.Size([3, 512])

# L2-normalise
image_embs_clip = image_embs_clip / image_embs_clip.norm(dim=-1, keepdim=True)
print("Normalised — all norms:", image_embs_clip.norm(dim=-1))

---
### 2.4 — Compute and Display the 3×3 Similarity Matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Compute similarity matrix: text (rows) vs images (cols)
# Both are already L2-normalised, so dot product = cosine similarity
sim_matrix = text_embs_clip @ image_embs_clip.T   # shape: (3, 3)
sim_np = sim_matrix.detach().cpu().numpy()

print("Similarity matrix (rows=captions, cols=images):")
print(f"{'':25}" + "".join(f"{l:>10}" for l in image_labels))
for i, cap_label in enumerate(["puppy caption", "cat caption", "car caption"]):
    row = "".join(f"{sim_np[i,j]:>10.3f}" for j in range(3))
    print(f"{cap_label:25}" + row)

# Plot as heatmap
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sim_np, cmap="Blues", vmin=0, vmax=0.5)

# Annotate each cell with the numeric score
for i in range(3):
    for j in range(3):
        text_colour = "white" if sim_np[i, j] > 0.3 else "black"
        ax.text(j, i, f"{sim_np[i, j]:.2f}", ha="center", va="center",
                color=text_colour, fontsize=12, fontweight="bold")

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(image_labels)
ax.set_yticklabels(["puppy text", "cat text", "car text"])
ax.set_title("CLIP Similarity Matrix\n(diagonal = matched pairs)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Verify diagonal is highest in each row
for i in range(3):
    best_match_idx = sim_np[i].argmax()
    print(f"Row {i} ({['puppy','cat','car'][i]} caption): best match = {image_labels[best_match_idx]} (score={sim_np[i,best_match_idx]:.3f})")

---
## Part 3 — CLIP via sentence-transformers

---
### 3.1 — Load the CLIP Model via SentenceTransformer

In [ ]:
from sentence_transformers import SentenceTransformer

# SentenceTransformer wraps CLIP with a simpler encode() API
# The model name "clip-ViT-B-32" corresponds to openai/clip-vit-base-patch32
st_model = SentenceTransformer("clip-ViT-B-32")

print("Model type:", type(st_model).__name__)

---
### 3.2 — Encode Images and Captions

In [ ]:
import numpy as np

# Encode images — pass PIL Image objects directly
# sentence-transformers handles preprocessing internally
st_image_embs = st_model.encode(images)    # shape: (3, 512)
print("Image embeddings shape:", st_image_embs.shape)

# Encode captions — pass list of strings
st_text_embs = st_model.encode(captions)   # shape: (3, 512)
print("Text embeddings shape: ", st_text_embs.shape)

---
### 3.3 — Compare Results to Part 2

In [ ]:
from sentence_transformers import util
import numpy as np

# Compute similarity matrix — util.cos_sim normalises internally
sim_matrix_st = util.cos_sim(st_text_embs, st_image_embs).numpy()

print("sentence-transformers similarity matrix:")
print(f"{'':25}" + "".join(f"{l:>10}" for l in image_labels))
for i, cap_label in enumerate(["puppy caption", "cat caption", "car caption"]):
    row = "".join(f"{sim_matrix_st[i,j]:>10.3f}" for j in range(3))
    print(f"{cap_label:25}" + row)

print()

# Compare to Part 2 (HuggingFace CLIPModel)
# Both should give the same values — they're the same underlying model
max_diff = np.abs(sim_np - sim_matrix_st).max()
print(f"Max absolute difference vs Part 2: {max_diff:.6f}")
print("(Expected < 0.01 — only floating point precision differences)")

---
## Part 4 — BLIP-2: Preprocessing

> **Note:** Parts 4 and 5 require GPU with ~16GB VRAM.

---
### 4.1 — Load the BLIP-2 Processor and Model

In [ ]:
from transformers import AutoProcessor, Blip2ForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Processor handles image resizing/normalisation AND text tokenization in one object
blip_processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")

# Load BLIP-2 in float16 to halve VRAM usage (2.7B params × 2 bytes ≈ 5.4 GB)
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16
)
blip_model.to(device)
blip_model.eval()

# Inspect the three frozen components
print("Vision model (frozen ViT):   ", type(blip_model.vision_model).__name__)
print("Language model (frozen OPT): ", type(blip_model.language_model).__name__)
print("Q-Former (trainable bridge): ", type(blip_model.qformer).__name__)

---
### 4.2 — Preprocess an Image

In [ ]:
from PIL import Image
from urllib.request import urlopen

CAR_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/2009_Audi_TT_--_10-30-2011.jpg/640px-2009_Audi_TT_--_10-30-2011.jpg"

# Load a wide image to demonstrate the aspect-ratio flattening
car_image = Image.open(urlopen(CAR_URL)).convert("RGB")
print(f"Original image size (W × H): {car_image.size}")

# The processor resizes to 224×224 regardless of input dimensions
# This is a hard constraint from the frozen ViT's pre-training resolution
inputs = blip_processor(car_image, return_tensors="pt")

print(f"pixel_values shape: {inputs['pixel_values'].shape}")
# → torch.Size([1, 3, 224, 224])
# 1 = batch, 3 = RGB channels, 224×224 = fixed resolution

# Why always 224×224?
# The frozen ViT inside BLIP-2 was pre-trained on 224×224 images.
# Changing the resolution would require re-training the ViT (which is frozen).
# 224×224 with patch_size=14 produces 16×16=196 patches — BLIP-2's ViT uses 14×14 patches.

---
### 4.3 — Visualise the Preprocessed Image

In [ ]:
import matplotlib.pyplot as plt
import torch

# Extract pixel_values and prepare for display
pixel_values = inputs["pixel_values"].cpu()  # shape: (1, 3, 224, 224)

# Remove batch dimension and rearrange channels: (3, 224, 224) → (224, 224, 3)
display_tensor = pixel_values[0].permute(1, 2, 0)   # now (H, W, C)

# Clamp to [0, 1] — normalised values can go slightly outside this range
display_tensor = display_tensor.clamp(0, 1)

# Side-by-side: original (resized to 224×224) vs preprocessed
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: original resized to match dimensions
original_resized = car_image.resize((224, 224))
ax1.imshow(original_resized)
ax1.set_title(f"Original (resized to 224×224)\nOriginal size: {car_image.size}")
ax1.axis("off")

# Right: after processor normalisation
ax2.imshow(display_tensor.numpy())
ax2.set_title("After BLIP-2 processor\n(normalised — note colour shift)")
ax2.axis("off")

plt.tight_layout()
plt.show()

print("The colour shift is from ImageNet normalisation:")
print(f"  pixel_values mean: {pixel_values.mean():.4f}  (should be ≈ 0)")
print(f"  pixel_values std:  {pixel_values.std():.4f}   (should be ≈ 1)")

In [ ]:
# VRAM cleanup between Parts 4 and 5
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("VRAM cleared")

---
## Part 5 — BLIP-2: Use Cases

---
### 5.1 — Image Captioning

In [ ]:
import torch

# Preprocess image — no text argument, so the LLM gets only visual context
inputs = blip_processor(car_image, return_tensors="pt").to(device, torch.float16)

# Generate caption autoregressively
# The LLM produces one token at a time until max_new_tokens is reached
with torch.no_grad():
    generated_ids = blip_model.generate(**inputs, max_new_tokens=20)

# Decode token IDs back to text, removing special tokens like </s> and <pad>
caption = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
print("Generated caption:", caption)

# Try a second image
cat_image = images[1]   # from Part 2
inputs_cat = blip_processor(cat_image, return_tensors="pt").to(device, torch.float16)

with torch.no_grad():
    generated_ids_cat = blip_model.generate(**inputs_cat, max_new_tokens=20)

caption_cat = blip_processor.batch_decode(generated_ids_cat, skip_special_tokens=True)[0].strip()
print("Cat image caption:", caption_cat)

---
### 5.2 — Visual Question Answering

In [ ]:
import torch

questions = [
    "What color is the car?",
    "What time of day does this appear to be?",
]

for question in questions:
    # Add the OPT-style Q/A prompt template
    prompt = f"Question: {question} Answer:"

    # Process image + text together
    # The processor packs them: pixel_values + input_ids in one dict
    inputs = blip_processor(
        car_image,
        text=prompt,
        return_tensors="pt"
    ).to(device, torch.float16)

    # Generate answer
    with torch.no_grad():
        generated_ids = blip_model.generate(**inputs, max_new_tokens=20)

    answer = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    print(f"Q: {question}")
    print(f"A: {answer}")
    print()

---
### 5.3 — Chat-Style Multi-Turn Prompting

In [ ]:
import torch

def blip2_generate(model, processor, image, prompt, device, max_new_tokens=30):
    """Run one BLIP-2 generation step and return the answer string."""
    # Process image and text prompt together
    inputs = processor(image, text=prompt, return_tensors="pt").to(device, torch.float16)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Decode and strip: take everything up to the first "Question" to avoid echoing history
    raw = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    answer = raw.split("Question")[0].strip()
    return answer


# Turn 1: initial question
q1 = "Write down what you see in this picture."
prompt = f"Question: {q1} Answer:"
a1 = blip2_generate(blip_model, blip_processor, car_image, prompt, device)
print(f"USER:   {q1}")
print(f"BLIP-2: {a1}")
print()

# Turn 2: include Turn 1 in the prompt before asking the follow-up
# The image (soft visual prompts) is constant — only the text context grows
q2 = "What would it cost to drive such a car?"
prompt = f"Question: {q1} Answer: {a1}. Question: {q2} Answer:"
a2 = blip2_generate(blip_model, blip_processor, car_image, prompt, device)
print(f"USER:   {q2}")
print(f"BLIP-2: {a2}")
print()

# Turn 3: include Turns 1 and 2 before the third question
q3 = "Why that much money?"
prompt = f"Question: {q1} Answer: {a1}. Question: {q2} Answer: {a2}. Question: {q3} Answer:"
a3 = blip2_generate(blip_model, blip_processor, car_image, prompt, device)
print(f"USER:   {q3}")
print(f"BLIP-2: {a3}")

---
### 5.4 — Theory Exercise: Conversation Prompt Formatter

In [ ]:
def format_blip2_prompt(history, new_question):
    """
    Build a BLIP-2 multi-turn conversation prompt from history.

    Args:
        history:      list of (question_str, answer_str) tuples
        new_question: string — the next question to ask

    Returns:
        prompt: string in BLIP-2 Q/A format, ending with "Answer:"
    """
    # Format each past exchange as "Question: {q} Answer: {a}."
    parts = [f"Question: {q} Answer: {a}." for q, a in history]

    # Append the new question at the end
    parts.append(f"Question: {new_question} Answer:")

    # Join with spaces
    return " ".join(parts)


# Test cases
result_1 = format_blip2_prompt([], "What is this?")
assert result_1 == "Question: What is this? Answer:", f"Got: {result_1!r}"

result_2 = format_blip2_prompt(
    [("What is this?", "A red car")],
    "What color is it?"
)
assert result_2 == "Question: What is this? Answer: A red car. Question: What color is it? Answer:", f"Got: {result_2!r}"

result_3 = format_blip2_prompt(
    [("What is this?", "A red car"), ("What color is it?", "Red")],
    "How fast can it go?"
)
expected_3 = "Question: What is this? Answer: A red car. Question: What color is it? Answer: Red. Question: How fast can it go? Answer:"
assert result_3 == expected_3, f"Got: {result_3!r}"

print("5.4 passed — all 3 test cases correct")
print()
print("Example output:")
print(result_2)

# Bonus: use format_blip2_prompt in a real conversation
print()
print("Using format_blip2_prompt in a real BLIP-2 conversation:")
conversation_history = []

for question in ["What do you see?", "What color is the main object?"]:
    prompt = format_blip2_prompt(conversation_history, question)
    answer = blip2_generate(blip_model, blip_processor, car_image, prompt, device)
    conversation_history.append((question, answer))
    print(f"USER:   {question}")
    print(f"BLIP-2: {answer}")
    print()